In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
import psycopg2
from pgvector.psycopg2 import register_vector

In [ ]:
# Config
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

EMBEDDING_MODEL = "gemini-embedding-001"

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "database": os.getenv("DB_NAME"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD")
}

def get_connection():
    conn = psycopg2.connect(**DB_CONFIG)
    register_vector(conn)
    return conn

In [ ]:
def retrieve(query: str, top_k: int = 5, category: str = None) -> list[dict]:
    """
    Retrieve top_k most similar chunks from PostgreSQL using dense vector search.
    
    Args:
        query: Query text to embed and search
        top_k: Number of top results to return
        category: Optional category filter
    
    Returns:
        List of dictionaries with chunk metadata and similarity score
    """
    # Embed query
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
    )
    query_embedding = response.embeddings[0].values
    
    # Build SQL query
    base_sql = '''
    SELECT chunk_id, parent_doc_id, category, chunk_title, summary, content,
           1 - (embedding <=> %s::vector) AS similarity
    FROM chunks
    '''
    
    if category:
        sql = base_sql + f"WHERE category = %s "
        params = [query_embedding, category]
    else:
        sql = base_sql
        params = [query_embedding]
    
    sql += "ORDER BY embedding <=> %s::vector LIMIT %s"
    params.extend([query_embedding, top_k])
    
    # Execute query
    results = []
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            rows = cur.fetchall()
            
            for row in rows:
                results.append({
                    'chunk_id': row[0],
                    'parent_doc_id': row[1],
                    'category': row[2],
                    'chunk_title': row[3],
                    'summary': row[4],
                    'content': row[5],
                    'similarity': row[6]
                })
    
    return results

In [ ]:
def pretty_print(results):
    """
    Print retrieval results in a formatted, readable way.
    
    Args:
        results: List of result dictionaries from retrieve()
    """
    for idx, result in enumerate(results, 1):
        print(f"[{idx}] {result['chunk_id']} | similarity: {result['similarity']:.2f}")
        if result['chunk_title']:
            print(f"    Title: {result['chunk_title']}")
        if result['summary']:
            print(f"    Summary: {result['summary'][:200]}...")
        print("    ---")